# 8. Evaluation

Per outlet, from the saved article outputs. The two approaches in the same two spaces:

| | levels | score |
|---|---|---|
| classifier | majority level | expected score (probabilities × mean training score of each level) |
| regressor | NewsGuard bounds · training quotas · article vote | mean score · score remapped to training quantiles |
| no text | most frequent level | mean training score |

95% intervals: bootstrap over outlets (fixed seed). Classifier vs regressor: paired, on the same outlets.

In [1]:
MODELS = "TF-IDF,Qwen3-8B,Mistral-24B,ModernBERT"
B = 2000

In [2]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import metrics as ME
import models as MO

art, outlets = MO.load()

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [3]:
VIEWS = {}                                         # (model, approach, view) -> (space, true, predicted)
for name in MODELS.split(","):
    pa = MO.predictions("classification", name)
    if pa is not None:
        T = ME.outlets_classifier(pa, outlets)
        VIEWS.update({(name, "classifier", v): x for v, x in ME.classifier_views(T, outlets).items()})
    pa = MO.predictions("regression", name)
    if pa is not None:
        T = ME.outlets_regressor(pa, outlets)
        VIEWS.update({(name, "regressor", v): x for v, x in ME.regressor_views(T, pa, art).items()})
VIEWS.update({("no text", "baseline", v): x for v, x in ME.baseline_views(outlets).items()})
rows = [{"model": m, "approach": a, "view": v, "space": s, **ME.METRICS[s](y, p)} for (m, a, v), (s, y, p) in VIEWS.items()]
all_metrics = pd.DataFrame(rows); all_metrics.to_csv(C.OUT / "all_metrics.csv", index=False)
print(f"{len(all_metrics)} views; models: {sorted({m for m, _, _ in VIEWS})}")

28 views; models: ['Mistral-24B', 'ModernBERT', 'Qwen3-8B', 'TF-IDF', 'no text']


In [4]:
all_metrics[all_metrics.space == "levels"].dropna(axis=1, how="all").set_index(["model", "approach", "view"]).round(3)

space  F1 macro  accuracy  balanced accuracy  tolerant (±1 level)  mean level error  kappa  quadratic kappa  Spearman  Kendall  F1 weighted  F1 l1  \
model       approach   view                                                                                                                                                                            
TF-IDF      classifier majority level            levels     0.340     0.416              0.343                0.796             0.876  0.202            0.529     0.533    0.451        0.383  0.394   
            regressor  levels: NewsGuard bounds  levels     0.233     0.305              0.301                0.827             0.872  0.137            0.479     0.597    0.520        0.276  0.053   
                       levels: training quotas   levels     0.376     0.407              0.377                0.845             0.770  0.231            0.681     0.671    0.560        0.407  0.423   
                       levels: article vote      levels     0.304     0.389              0.324                0.823             0.850  0.170            0.596     0.569    0.482        0.351  0.451   
Qwen3-8B    classifier majority level            levels     0.384     0.412              0.391                0.819             0.801  0.239            0.671     0.682    0.575        0.408  0.471   
            regressor  levels: NewsGuard bounds  levels     0.256     0.367              0.333                0.881             0.752  0.185            0.606     0.692    0.606        0.310  0.000   
                       levels: training quotas   levels     0.411     0.429              0.411                0.881             0.708  0.259            0.722     0.715    0.607        0.429  0.507   
                       levels: article vote      levels     0.319     0.389              0.335                0.845             0.796  0.185            0.668     0.653    0.555        0.363  0.438   
Mistral-24B classifier majority level            levels     0.400     0.434              0.401                0.819             0.819  0.255            0.596     0.599    0.515        0.426  0.533   
            regressor  levels: NewsGuard bounds  levels     0.282     0.372              0.341                0.881             0.748  0.182            0.621     0.685    0.603        0.323  0.150   
                       levels: training quotas   levels     0.425     0.442              0.426                0.863             0.717  0.277            0.705     0.703    0.599        0.442  0.507   
                       levels: article vote      levels     0.387     0.447              0.399                0.854             0.726  0.265            0.693     0.690    0.593        0.425  0.471   
ModernBERT  regressor  levels: NewsGuard bounds  levels     0.251     0.358              0.312                0.810             0.841  0.166            0.499     0.559    0.481        0.309  0.051   
                       levels: training quotas   levels     0.377     0.389              0.379                0.810             0.858  0.208            0.596     0.589    0.498        0.388  0.535   
                       levels: article vote      levels     0.305     0.350              0.309                0.770             0.947  0.128            0.544     0.538    0.453        0.332  0.444   
no text     baseline   most frequent level       levels     0.104     0.350              0.200                0.655             1.150  0.000            0.000       NaN      NaN        0.181  0.000   

                                                 F1 l2  F1 l3  F1 l4  F1 l5  reliable: accuracy  reliable: F1 macro  reliable: MCC  
model       approach   view                                                                                                         
TF-IDF      classifier majority level            0.324  0.205  0.559  0.219               0.757               0.721          0.446  
            regressor  levels: NewsGuard bounds  0

In [5]:
all_metrics[all_metrics.space == "score"].dropna(axis=1, how="all").set_index(["model", "approach", "view"]).round(3)

space  Spearman  Kendall  reliable: accuracy  reliable: F1 macro     MAE    RMSE  median error     R²  Pearson  Lin's CCC  bias (predicted − true)  \
model       approach   view                                                                                                                                                                      
TF-IDF      classifier expected score       score     0.648    0.460               0.779               0.732  17.189  21.654        15.279  0.359    0.630      0.461                   -0.014   
            regressor  mean score           score     0.674    0.480               0.788               0.748  17.825  21.987        16.192  0.339    0.644      0.422                   -1.404   
                       remapped score       score     0.675    0.495               0.819               0.801  16.887  22.404        13.000  0.314    0.665      0.665                   -0.400   
Qwen3-8B    classifier expected score       score     0.698    0.499               0.814               0.794  16.300  20.412        14.120  0.430    0.684      0.545                   -1.462   
            regressor  mean score           score     0.711    0.514               0.823               0.802  16.311  20.297        13.510  0.437    0.688      0.550                   -0.889   
                       remapped score       score     0.719    0.539               0.827               0.811  15.172  21.412        11.591  0.373    0.694      0.694                   -0.400   
Mistral-24B classifier expected score       score     0.710    0.523               0.805               0.786  15.311  19.449        13.513  0.483    0.709      0.610                   -0.578   
            regressor  mean score           score     0.735    0.540               0.823               0.803  15.793  19.597        13.971  0.475    0.717      0.588                   -0.918   
                       remapped score       score     0.732    0.554               0.819               0.801  15.227  20.896        12.500  0.403    0.709      0.709                   -0.400   
ModernBERT  regressor  mean score           score     0.610    0.435               0.761               0.721  17.542  21.958        14.925  0.341    0.591      0.473                    0.060   
                       remapped score       score     0.613    0.454               0.765               0.743  17.694  23.902        12.636  0.219    0.619      0.619                   -0.400   
no text     baseline   mean training score  score    -0.015   -0.012               0.650               0.394  22.575  27.049        19.175 -0.000   -0.014     -0.000                   -0.001   

                                            within 10 points  within 20 points  reliable: AUC  
model       approach   view                                                                    
TF-IDF      classifier expected score                  0.354             0.646          0.842  
            regressor  mean score                      0.332             0.615          0.860  
                       remapped score                  0.381             0.677          0.859  
Qwen3-8B    classifier expected score                  0.332             0.664          0.871  
            regressor  mean score                      0.341             0.699          0.880  
                       remapped score                  0.478             0.739          0.880  
Mistral-24B classifier expected score                  0.412             0.717          0.854  
            regressor  mean score                      0.367             0.681          0.876  
                       remapped score                  0.438             0.735          0.874  
ModernBERT  regressor  mean score                      0.341             0.637          0.835  
                       remapped score                  0.385             0.681          0.833  
no text     baseline   mean training score             0.181             0.509 

## Intervals

In [6]:
KEYS = {"levels": ["F1 macro", "tolerant (±1 level)", "quadratic kappa", "Spearman", "reliable: accuracy"],
        "score": ["MAE", "Spearman", "within 10 points", "within 20 points", "reliable: accuracy"]}
rows = []
for (m, a, v), (s, y, p) in VIEWS.items():
    ci, value = ME.bootstrap(y, p, s, KEYS[s], B=B), ME.METRICS[s](y, p)
    rows += [{"model": m, "approach": a, "view": v, "space": s, "metric": c, "value": value[c], "low": ci[c][0],
              "high": ci[c][1]} for c in KEYS[s]]
intervals = pd.DataFrame(rows); intervals.to_csv(C.OUT / "intervals.csv", index=False)
print(len(intervals), "intervals")

140 intervals


## Classifier vs regressor (regressor − classifier, same outlets)

In [7]:
PAIRS = {"levels": ("majority level", ["levels: NewsGuard bounds", "levels: training quotas", "levels: article vote"]),
         "score": ("expected score", ["mean score", "remapped score"])}
rows = []
for name in MODELS.split(","):
    for space, (cls_view, reg_views) in PAIRS.items():
        if (name, "classifier", cls_view) not in VIEWS or (name, "regressor", reg_views[0]) not in VIEWS:
            continue
        for reg_view in reg_views:
            for key in KEYS[space]:
                d, (lo, hi) = ME.bootstrap_difference(VIEWS[(name, "regressor", reg_view)], VIEWS[(name, "classifier", cls_view)], key, B=B)
                rows.append({"model": name, "space": space, "regressor view": reg_view, "metric": key,
                             "difference": d, "low": lo, "high": hi, "not chance": not (lo <= 0 <= hi)})
pairs = pd.DataFrame(rows); pairs.to_csv(C.OUT / "classifier_vs_regressor.csv", index=False)
pairs.round(3)

,model,space,regressor view,metric,difference,low,high,not chance
0,TF-IDF,levels,levels: NewsGuard bounds,F1 macro,-0.107,-0.180,-0.030,True
1,TF-IDF,levels,levels: NewsGuard bounds,tolerant (±1 level),0.031,-0.031,0.093,False
2,TF-IDF,levels,levels: NewsGuard bounds,quadratic kappa,-0.050,-0.137,0.041,False
3,TF-IDF,levels,levels: NewsGuard bounds,Spearman,0.064,-0.017,0.150,False
4,TF-IDF,levels,levels: NewsGuard bounds,reliable: accuracy,0.035,-0.013,0.080,False
...,...,...,...,...,...,...,...,...
70,Mistral-24B,score,remapped score,MAE,-0.084,-1.610,1.443,False
71,Mistral-24B,score,remapped score,Spearman,0.022,-0.010,0.054,False
72,Mistral-24B,score,remapped score,within 10 points,0.027,-0.049,0.102,False
73,Mistral-24B,score,remapped score,within 20 points,0.018,-0.035,0.075,False


## Probability metrics of the classifiers

In [8]:
pd.DataFrame({name: ME.probability_metrics(ME.outlets_classifier(MO.predictions("classification", name), outlets))
              for name in MODELS.split(",") if MO.predictions("classification", name) is not None}).T.round(3)

,AUC macro (5 levels),log loss,Brier,reliable: AUC (probabilities)
TF-IDF,0.678,1.459,0.735,0.826
Qwen3-8B,0.724,1.358,0.709,0.843
Mistral-24B,0.719,1.391,0.710,0.832
